# 基于傅里叶变换的低通、高通、带通频域滤波去噪实验：学生练习版

本实验使用 `numpy.fft` 完成傅里叶变换和反变换。练习版保留网络图像读取、频谱显示、鼠标点击亮点、结果展示和对比分析框架，将频域滤波与频谱亮点擦除中的关键算法设置为 `TODO`，需要学生补全。

需要完成的核心内容：

1. 根据点击点生成圆形频谱擦除掩膜。
2. 擦除频谱亮点并反向傅里叶变换。
3. 计算频率距离矩阵。
4. 手写理想低通、高通、带通滤波器。
5. 手写巴特沃斯低通、高通、带通滤波器。
6. 将频域滤波器作用到频谱并反变换。

## 1. 相关背景知识

### 1.1 图像频率域

图像可以看作二维信号。傅里叶变换可以将图像从空间域转换到频率域：

1. **低频成分**：图像中变化缓慢的部分，例如整体亮度、大面积平滑区域。
2. **高频成分**：图像中变化剧烈的部分，例如边缘、纹理、细小噪声。
3. **中间频率成分**：介于低频和高频之间，可能对应某些纹理、结构或周期性变化。

### 1.2 频域滤波基本思想

频域滤波通常包含以下步骤：

1. 对图像进行二维傅里叶变换，得到频率域复数矩阵。
2. 将频谱中心化，使低频移动到中心位置。
3. 构造滤波器掩膜。
4. 将频谱与滤波器逐点相乘。
5. 将滤波后的频谱反中心化。
6. 进行傅里叶反变换，得到空间域图像。

### 1.3 三类滤波器

**低通滤波器**：保留低频，抑制高频。常用于平滑图像、抑制高频噪声，但会模糊边缘。

**高通滤波器**：保留高频，抑制低频。常用于突出边缘和纹理，但也可能增强噪声。

**带通滤波器**：只保留某一频率范围内的成分，抑制过低和过高的频率。可以用于观察特定频率结构，也可在某些情况下抑制不需要的频率成分。

### 1.4 理想滤波器与巴特沃斯滤波器

理想滤波器的掩膜通常只有 `0` 和 `1` 两种取值。例如，理想低通滤波器会让截止半径以内的频率完全通过，让截止半径以外的频率完全抑制。这种方法频率选择非常明确，但在截止频率处存在突然跳变，容易导致空间域图像出现振铃或边缘附近的波纹现象。

巴特沃斯滤波器是一类具有平滑过渡特性的频域滤波器。它不会在截止半径处从 `1` 突然变成 `0`，而是在截止频率附近逐渐变化，因此空间域结果通常更柔和。

巴特沃斯滤波器有两个重要参数：

1. `D0`：截止半径，控制保留或抑制频率的范围。
2. `n`：滤波器阶数，控制过渡带的陡峭程度。阶数越高，巴特沃斯滤波器越接近理想滤波器。

### 1.5 巴特沃斯低通滤波器

巴特沃斯低通滤波器用于保留低频、抑制高频，其传递函数为：

$$
H_{LP}(D)=\frac{1}{1+(D/D_0)^{2n}}
$$

其中：

1. `D` 表示当前频率点到频谱中心的距离。
2. `D0` 表示低通截止半径。
3. `n` 表示滤波器阶数。

当 `D` 很小时，`H_{LP}(D)` 接近 `1`，低频被保留；当 `D` 很大时，`H_{LP}(D)` 接近 `0`，高频被抑制。

### 1.6 巴特沃斯高通滤波器

巴特沃斯高通滤波器用于抑制低频、保留高频，可以由低通滤波器得到：

$$
H_{HP}(D)=1-H_{LP}(D)
$$

也可以写成：

$$
H_{HP}(D)=\frac{1}{1+(D_0/D)^{2n}}
$$

高通滤波器能够突出边缘、纹理和细节，但也可能增强噪声。因此，高通滤波结果通常更适合用于观察高频成分，而不一定适合作为最终去噪图像。

### 1.7 巴特沃斯带通滤波器

巴特沃斯带通滤波器只保留某一段频率范围，抑制过低和过高的频率。本实验中使用两个巴特沃斯低通滤波器相减构造带通滤波器：

$$
H_{BP}(D)=H_{LP}(D;D_{high})-H_{LP}(D;D_{low})
$$

其中：

1. `D_low` 表示低截止半径。
2. `D_high` 表示高截止半径。
3. 只有位于两个截止半径之间的频率成分会被较多保留。

带通滤波器适合观察中间频率结构、纹理成分或某些周期性成分。与理想带通滤波器相比，巴特沃斯带通滤波器在两个截止半径附近都有平滑过渡。

## 2. 实验步骤

本实验按照以下步骤完成：

1. 构造一张灰度测试图像。
2. 为图像添加随机噪声和周期性噪声。
3. 使用 `np.fft.fft2` 计算二维傅里叶变换。
4. 使用 `np.fft.fftshift` 将频谱中心化。
5. 手写频率距离矩阵。
6. 手写低通、高通、带通滤波器。
7. 将滤波器作用到频谱上。
8. 使用 `np.fft.ifftshift` 和 `np.fft.ifft2` 还原图像。
9. 对比滤波前后图像、频谱和误差指标。

## 学生练习任务与代码补全步骤

请按照下面顺序补全代码。建议每补完一个函数，就运行对应单元检查结果。

### 1. 补全 `create_circular_erase_mask`

目标：根据频谱图上的点击坐标，生成圆形擦除掩膜。

实现步骤：

1. 根据 `shape` 获取频谱高度 `h` 和宽度 `w`。
2. 计算频谱中心 `(cy, cx)`。
3. 使用 `np.mgrid` 生成所有频率点坐标 `yy, xx`。
4. 初始化全 1 掩膜 `mask`，表示默认保留所有频率。
5. 遍历用户点击的点 `(y, x)`。
6. 如果 `erase_symmetric=True`，同时计算该点关于中心的对称点：
   - `sy = 2 * cy - y`
   - `sx = 2 * cx - x`
7. 对每个要擦除的点，生成圆形区域：

$$
(yy-y)^2+(xx-x)^2 \le radius^2
$$

8. 将圆形区域对应的 `mask` 置为 `0`。
9. 返回 `mask`。

### 2. 补全 `erase_spectrum_points_and_reconstruct`

目标：擦除中心化频谱中的亮点，再反向傅里叶变换。

实现步骤：

1. 调用 `fft2_centered(image)` 得到中心化频谱。
2. 调用 `create_circular_erase_mask` 得到擦除掩膜。
3. 将频谱与掩膜逐点相乘。
4. 调用 `ifft2_from_centered` 将修改后的频谱还原到空间域。
5. 使用 `np.clip` 将图像限制在 0 到 1。
6. 返回重建图像、修改后的频谱和擦除掩膜。

### 3. 补全 `create_frequency_distance`

目标：为频域滤波器计算每个频率点到中心的距离。

实现步骤：

1. 获取频谱高度 `h` 和宽度 `w`。
2. 计算中心点 `(cy, cx)`。
3. 使用 `np.mgrid` 生成坐标矩阵。
4. 根据欧氏距离公式计算：

$$
D(y,x)=\sqrt{(y-cy)^2+(x-cx)^2}
$$

5. 返回距离矩阵。

### 4. 补全理想低通、高通、带通滤波器

理想滤波器掩膜只有 0 和 1。

1. 低通：`distance <= cutoff` 的位置为 1。
2. 高通：`distance >= cutoff` 的位置为 1。
3. 带通：`low_cutoff <= distance <= high_cutoff` 的位置为 1。
4. 注意检查 `low_cutoff < high_cutoff`。

### 5. 补全巴特沃斯低通、高通、带通滤波器

1. 巴特沃斯低通：

$$
H_{LP}(D)=\frac{1}{1+(D/D_0)^{2n}}
$$

2. 巴特沃斯高通：

$$
H_{HP}(D)=1-H_{LP}(D)
$$

3. 巴特沃斯带通：

$$
H_{BP}=H_{LP}(D_{high})-H_{LP}(D_{low})
$$

4. 使用 `np.clip(mask, 0, 1)` 避免数值越界。

### 6. 补全 `apply_frequency_filter`

目标：将频域滤波器作用到图像上。

实现步骤：

1. 调用 `fft2_centered(image)` 得到中心化频谱。
2. 将频谱与 `mask` 逐点相乘。
3. 调用 `ifft2_from_centered` 反变换回空间域。
4. 使用 `np.clip` 限制灰度范围。
5. 返回滤波后的图像和滤波后的频谱。

### 7. 最终检查

1. 先运行网络图像读取和傅里叶变换单元。
2. 如果鼠标交互不可用，可以手动设置 `selected_spectrum_points`。
3. 补全频谱擦除函数后，观察擦除前后频谱和图像变化。
4. 补全滤波器函数后，观察理想滤波器和巴特沃斯滤波器掩膜差异。
5. 最后运行全部单元，比较低通、高通、带通以及理想/巴特沃斯结果差异。

## 3. 导入基础库

In [ ]:
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 4. 从网络读取灰度图像



In [ ]:
IMAGE_URLS = [
    "https://images2015.cnblogs.com/blog/890966/201612/890966-20161227132710523-1258992576.jpg",
]


def load_grayscale_image_from_url(url, target_size=160):
    """
    从网络地址读取图像，并转换为灰度图像。

    参数：
        url: 网络图像地址
        target_size: 缩放后的长边大小

    返回：
        gray_image: float64 灰度图像，范围 0 到 1
    """
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"},
        verify=False,
    )
    response.raise_for_status()

    image = Image.open(BytesIO(response.content)).convert("L")

    gray_image = np.array(image, dtype=np.float64) / 255.0
    return gray_image


def load_first_available_grayscale_image(urls, target_size=160):
    """
    依次尝试多个网络图像地址，返回第一张成功读取的灰度图像。
    """
    errors = []
    for url in urls:
        try:
            image = load_grayscale_image_from_url(url, target_size=target_size)
            return image, url
        except Exception as exc:
            errors.append(f"{url} -> {exc}")
    raise RuntimeError("所有网络图像读取失败：\n" + "\n".join(errors))


noisy_image, used_image_url = load_first_available_grayscale_image(IMAGE_URLS, target_size=160)

print("网络图像读取成功：", used_image_url)
print("灰度图像尺寸：", noisy_image.shape)
print("灰度范围：", float(noisy_image.min()), "到", float(noisy_image.max()))

plt.figure(figsize=(5, 5))
plt.imshow(noisy_image, cmap="gray", vmin=0, vmax=1)
plt.title("网络灰度图像")
plt.axis("off")
plt.show()

## 5. 使用 numpy.fft 进行傅里叶变换

In [ ]:
def fft2_centered(image):
    """
    使用 numpy.fft 计算中心化二维傅里叶变换。

    参数：
        image: 输入灰度图像

    返回：
        shifted_spectrum: 中心化频谱
    """
    spectrum = np.fft.fft2(image)
    shifted_spectrum = np.fft.fftshift(spectrum)
    return shifted_spectrum


def ifft2_from_centered(shifted_spectrum):
    """
    将中心化频谱反变换回空间域图像。

    参数：
        shifted_spectrum: 中心化频谱

    返回：
        image: 反变换后的实数图像
    """
    spectrum = np.fft.ifftshift(shifted_spectrum)
    image_complex = np.fft.ifft2(spectrum)
    image = np.real(image_complex)
    return image


def log_spectrum(shifted_spectrum):
    """
    计算对数幅度谱，便于观察。
    """
    return np.log1p(np.abs(shifted_spectrum))


noisy_spectrum = fft2_centered(noisy_image)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(noisy_image, cmap="gray", vmin=0, vmax=1)
plt.title("加噪图像")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(log_spectrum(noisy_spectrum), cmap="gray")
plt.title("中心化频谱")
plt.axis("off")

plt.tight_layout()
plt.show()

## 5.1 在中心化频谱图上点击亮点并擦除

周期性噪声在频率域中常表现为中心点以外的若干个亮点。下面增加一个交互功能：

1. 在中心化频谱图上用鼠标左键点击多个亮点。
2. 程序会以每个点击位置为圆心，生成圆形擦除区域。
3. 将这些亮点对应的频率系数置为 `0`。
4. 对修改后的频谱进行反向傅里叶变换，观察空间域图像变化。

注意：

1. 亮点坐标格式在程序中保存为 `(y, x)`，也就是 `(行, 列)`。
2. 对真实灰度图像，频谱通常具有共轭对称性；因此程序默认会同时擦除点击点关于频谱中心的对称点。
3. 如果当前 Jupyter 环境不支持鼠标点击，可以在后面的单元中手动填写 `selected_spectrum_points`。

In [ ]:
try:
    # JupyterLab / Notebook 中推荐使用 widget 后端，以支持鼠标点击。
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    try:
        get_ipython().run_line_magic("matplotlib", "notebook")
    except Exception:
        pass


class SpectrumPointSelector:
    """
    中心化频谱亮点选择器。

    左键点击：添加一个要擦除的频谱亮点。
    右键点击：删除距离鼠标最近的已选亮点。
    按 c：清空所有已选亮点。
    """

    def __init__(self, shifted_spectrum):
        self.shifted_spectrum = shifted_spectrum
        self.points = []

        self.fig, self.ax = plt.subplots(figsize=(7, 6))
        self.ax.imshow(log_spectrum(shifted_spectrum), cmap="gray")
        self.ax.set_title("左键添加亮点；右键删除最近点；按 c 清空")
        self.ax.axis("off")
        self.artist = self.ax.scatter([], [], c="red", s=60, marker="o", facecolors="none", linewidths=1.8)

        self.cid_click = self.fig.canvas.mpl_connect("button_press_event", self.on_click)
        self.cid_key = self.fig.canvas.mpl_connect("key_press_event", self.on_key)

    def update_artist(self):
        if self.points:
            ys, xs = zip(*self.points)
            self.artist.set_offsets(np.column_stack([xs, ys]))
        else:
            self.artist.set_offsets(np.empty((0, 2)))
        self.fig.canvas.draw_idle()

    def on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None or event.ydata is None:
            return

        x = int(round(event.xdata))
        y = int(round(event.ydata))
        h, w = self.shifted_spectrum.shape

        if x < 0 or x >= w or y < 0 or y >= h:
            return

        if event.button == 1:
            self.points.append((y, x))
            print(f"添加频谱亮点：y={y}, x={x}")
        elif event.button == 3 and self.points:
            distances = [(py - y) ** 2 + (px - x) ** 2 for py, px in self.points]
            remove_index = int(np.argmin(distances))
            removed = self.points.pop(remove_index)
            print(f"删除频谱亮点：y={removed[0]}, x={removed[1]}")

        self.update_artist()

    def on_key(self, event):
        if event.key == "c":
            self.points.clear()
            self.update_artist()
            print("已清空所有频谱亮点")

    def get_points(self):
        return list(self.points)


spectrum_selector = SpectrumPointSelector(noisy_spectrum)
plt.show()

## 5.2 根据点击点生成圆形擦除掩膜并反变换

运行下面单元会读取上一个单元点击得到的亮点坐标。如果没有成功使用鼠标交互，可以手动设置：

```python
selected_spectrum_points = [
    (30, 80),
    (120, 80),
]
```

其中每个点为 `(y, x)`。

In [ ]:
def create_circular_erase_mask(shape, points, radius=4, erase_symmetric=True):
    """
    TODO：根据频谱亮点坐标生成圆形擦除掩膜。

    参数：
        shape: 中心化频谱大小
        points: 要擦除的亮点坐标列表，每个点为 (y, x)
        radius: 圆形擦除区域半径
        erase_symmetric: 是否同时擦除关于频谱中心的对称点

    返回：
        mask: 频域掩膜，保留区域为 1，擦除区域为 0
    """
    # TODO 1：获取频谱大小 h, w。
    # TODO 2：计算频谱中心 cy, cx。
    # TODO 3：使用 np.mgrid 生成 yy, xx 坐标矩阵。
    # TODO 4：创建全 1 掩膜 mask。
    # TODO 5：遍历 points，把每个点击点加入 erase_points。
    # TODO 6：如果 erase_symmetric=True，计算并加入对称点。
    # TODO 7：对 erase_points 中每个点生成圆形区域，并将 mask 对应区域置为 0。
    # TODO 8：返回 mask。
    raise NotImplementedError("请补全 create_circular_erase_mask 函数")


def erase_spectrum_points_and_reconstruct(image, points, radius=4, erase_symmetric=True):
    """
    TODO：擦除中心化频谱中的若干圆形区域，并通过反向傅里叶变换重建图像。

    参数：
        image: 输入灰度图像
        points: 要擦除的频谱亮点坐标
        radius: 圆形擦除半径
        erase_symmetric: 是否同时擦除对称点

    返回：
        reconstructed: 反变换图像
        edited_spectrum: 擦除亮点后的中心化频谱
        erase_mask: 圆形擦除掩膜
    """
    # TODO 1：调用 fft2_centered(image) 得到中心化频谱 shifted_spectrum。
    # TODO 2：调用 create_circular_erase_mask 得到 erase_mask。
    # TODO 3：edited_spectrum = shifted_spectrum * erase_mask。
    # TODO 4：调用 ifft2_from_centered(edited_spectrum) 得到 reconstructed。
    # TODO 5：使用 np.clip 将 reconstructed 限制到 0 到 1。
    # TODO 6：返回 reconstructed, edited_spectrum, erase_mask。
    raise NotImplementedError("请补全 erase_spectrum_points_and_reconstruct 函数")


if "spectrum_selector" in globals():
    selected_spectrum_points = spectrum_selector.get_points()
else:
    selected_spectrum_points = []

# 如果鼠标交互不可用，可以取消下面注释并手动填写频谱亮点坐标。
# selected_spectrum_points = [
#     (30, 80),
#     (120, 80),
# ]

notch_radius = 4
erase_symmetric_points = True

erased_image, erased_spectrum, erase_mask = erase_spectrum_points_and_reconstruct(
    noisy_image,
    selected_spectrum_points,
    radius=notch_radius,
    erase_symmetric=erase_symmetric_points,
)

print("已选择频谱亮点数量：", len(selected_spectrum_points))
print("频谱亮点坐标：", selected_spectrum_points)
print("圆形擦除半径：", notch_radius)
print("是否同时擦除对称点：", erase_symmetric_points)

In [ ]:
plt.figure(figsize=(14, 8))

plt.subplot(2, 3, 1)
plt.imshow(noisy_image, cmap="gray", vmin=0, vmax=1)
plt.title("网络灰度图像")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(log_spectrum(noisy_spectrum), cmap="gray")
if selected_spectrum_points:
    ys, xs = zip(*selected_spectrum_points)
    plt.scatter(xs, ys, c="red", s=45, marker="o", facecolors="none", linewidths=1.5)
plt.title("点击选择的频谱亮点")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(erase_mask, cmap="gray", vmin=0, vmax=1)
plt.title("圆形擦除掩膜")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(log_spectrum(erased_spectrum), cmap="gray")
plt.title("擦除亮点后的频谱")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(erased_image, cmap="gray", vmin=0, vmax=1)
plt.title("擦除亮点后反变换图像")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(np.abs(noisy_image - erased_image), cmap="hot")
plt.title("擦除前后差异")
plt.axis("off")

plt.tight_layout()
plt.show()

print("擦除亮点后平均变化量：", mean_absolute_change(noisy_image, erased_image) if "mean_absolute_change" in globals() else np.mean(np.abs(noisy_image - erased_image)))

## 6. 手写低通、高通、带通滤波器

下面的滤波器不调用现成滤波函数，而是根据频率点到频谱中心的距离自己生成掩膜。本实验同时实现两类滤波器：

1. **理想滤波器**：在截止半径处直接从 1 跳变到 0，掩膜边界很硬。
2. **巴特沃斯滤波器**：在截止半径附近平滑过渡，掩膜变化更柔和。

设频谱中心为 `(cy, cx)`，频率点为 `(y, x)`，则距离为：

$$
D(y,x)=\sqrt{(y-cy)^2+(x-cx)^2}
$$

理想低通、高通、带通滤波器分别为：

1. 低通：`D <= cutoff`
2. 高通：`D >= cutoff`
3. 带通：`low_cutoff <= D <= high_cutoff`

巴特沃斯低通滤波器为：

$$
H_{LP}(D)=\frac{1}{1+(D/D_0)^{2n}}
$$

其中 `D0` 是截止半径，`n` 是滤波器阶数。高通可由 `1 - 低通` 得到，带通可由两个不同截止半径的低通相减得到。

In [ ]:
def create_frequency_distance(shape):
    """
    TODO：计算每个频率点到频谱中心的距离。

    参数：
        shape: 图像或频谱大小，形如 (h, w)

    返回：
        distance: 每个频率点到中心的欧氏距离
    """
    # TODO 1：获取 h, w。
    # TODO 2：计算中心点 cy, cx。
    # TODO 3：使用 np.mgrid 生成 yy, xx。
    # TODO 4：根据欧氏距离公式计算 distance。
    # TODO 5：返回 distance。
    raise NotImplementedError("请补全 create_frequency_distance 函数")


def low_pass_filter(shape, cutoff):
    """
    TODO：手写理想低通滤波器。
    """
    # TODO：distance <= cutoff 的位置为 1，其余为 0。
    raise NotImplementedError("请补全 low_pass_filter 函数")


def high_pass_filter(shape, cutoff):
    """
    TODO：手写理想高通滤波器。
    """
    # TODO：distance >= cutoff 的位置为 1，其余为 0。
    raise NotImplementedError("请补全 high_pass_filter 函数")


def band_pass_filter(shape, low_cutoff, high_cutoff):
    """
    TODO：手写理想带通滤波器。
    """
    # TODO 1：检查 low_cutoff 是否小于 high_cutoff。
    # TODO 2：计算 distance。
    # TODO 3：low_cutoff <= distance <= high_cutoff 的位置为 1，其余为 0。
    # TODO 4：返回 mask。
    raise NotImplementedError("请补全 band_pass_filter 函数")


def butterworth_low_pass_filter(shape, cutoff, order=2):
    """
    TODO：手写巴特沃斯低通滤波器。

    参数：
        shape: 频谱大小
        cutoff: 截止半径
        order: 滤波器阶数，阶数越大，过渡越陡

    返回：
        mask: 巴特沃斯低通滤波器掩膜
    """
    # TODO 1：计算 distance。
    # TODO 2：根据 1 / (1 + (distance / cutoff) ** (2 * order)) 计算 mask。
    # TODO 3：注意 cutoff 可以加一个很小的数避免除 0。
    # TODO 4：返回 mask。
    raise NotImplementedError("请补全 butterworth_low_pass_filter 函数")


def butterworth_high_pass_filter(shape, cutoff, order=2):
    """
    TODO：手写巴特沃斯高通滤波器。
    """
    # TODO：使用 1 - butterworth_low_pass_filter(...) 得到高通掩膜。
    raise NotImplementedError("请补全 butterworth_high_pass_filter 函数")


def butterworth_band_pass_filter(shape, low_cutoff, high_cutoff, order=2):
    """
    TODO：手写巴特沃斯带通滤波器。

    这里用两个巴特沃斯低通滤波器相减，得到一个平滑过渡的带通掩膜。
    """
    # TODO 1：检查 low_cutoff 是否小于 high_cutoff。
    # TODO 2：计算 low_removed = butterworth_low_pass_filter(shape, low_cutoff, order)。
    # TODO 3：计算 high_kept = butterworth_low_pass_filter(shape, high_cutoff, order)。
    # TODO 4：mask = high_kept - low_removed。
    # TODO 5：使用 np.clip(mask, 0, 1) 限制范围。
    # TODO 6：返回 mask。
    raise NotImplementedError("请补全 butterworth_band_pass_filter 函数")

## 7. 应用频域滤波器

In [ ]:
def apply_frequency_filter(image, mask):
    """
    TODO：对图像应用频域滤波器。

    参数：
        image: 输入空间域灰度图像
        mask: 频域滤波器掩膜

    返回：
        filtered_image: 滤波后的空间域图像
        filtered_spectrum: 滤波后的中心化频谱
    """
    # TODO 1：调用 fft2_centered(image) 得到 shifted_spectrum。
    # TODO 2：filtered_spectrum = shifted_spectrum * mask。
    # TODO 3：调用 ifft2_from_centered(filtered_spectrum) 得到 filtered_image。
    # TODO 4：使用 np.clip 将 filtered_image 限制到 0 到 1。
    # TODO 5：返回 filtered_image, filtered_spectrum。
    raise NotImplementedError("请补全 apply_frequency_filter 函数")


shape = noisy_image.shape

# 理想滤波器：作为对照。
ideal_low_mask = low_pass_filter(shape, cutoff=24)
ideal_high_mask = high_pass_filter(shape, cutoff=24)
ideal_band_mask = band_pass_filter(shape, low_cutoff=10, high_cutoff=42)

ideal_low_result, ideal_low_spectrum = apply_frequency_filter(noisy_image, ideal_low_mask)
ideal_high_result, ideal_high_spectrum = apply_frequency_filter(noisy_image, ideal_high_mask)
ideal_band_result, ideal_band_spectrum = apply_frequency_filter(noisy_image, ideal_band_mask)

# 巴特沃斯滤波器：作为本节主要滤波结果。
low_mask = butterworth_low_pass_filter(shape, cutoff=24, order=2)
high_mask = butterworth_high_pass_filter(shape, cutoff=24, order=2)
band_mask = butterworth_band_pass_filter(shape, low_cutoff=10, high_cutoff=42, order=2)

low_result, low_spectrum = apply_frequency_filter(noisy_image, low_mask)
high_result, high_spectrum = apply_frequency_filter(noisy_image, high_mask)
band_result, band_spectrum = apply_frequency_filter(noisy_image, band_mask)

## 8. 显示滤波器掩膜与滤波结果

In [ ]:
plt.figure(figsize=(14, 9))

filters = [
    ("巴特沃斯低通滤波器", low_mask, low_result, low_spectrum),
    ("巴特沃斯高通滤波器", high_mask, high_result, high_spectrum),
    ("巴特沃斯带通滤波器", band_mask, band_result, band_spectrum),
]

for i, (name, mask, result, spectrum) in enumerate(filters):
    plt.subplot(3, 4, i * 4 + 1)
    plt.imshow(mask, cmap="gray", vmin=0, vmax=1)
    plt.title(f"{name}掩膜")
    plt.axis("off")

    plt.subplot(3, 4, i * 4 + 2)
    plt.imshow(log_spectrum(spectrum), cmap="gray")
    plt.title("滤波后频谱")
    plt.axis("off")

    plt.subplot(3, 4, i * 4 + 3)
    plt.imshow(result, cmap="gray", vmin=0, vmax=1)
    plt.title("滤波结果")
    plt.axis("off")

    plt.subplot(3, 4, i * 4 + 4)
    plt.imshow(np.abs(noisy_image - result), cmap="hot")
    plt.title("与网络图像差异")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 9. 网络图像与三类滤波结果对比

In [ ]:
def mean_absolute_change(a, b):
    """
    计算两幅图像之间的平均绝对变化量。

    这里没有干净真值图像，因此该指标只表示滤波改变了多少像素强度，
    不能直接等价为去噪质量。
    """
    return np.mean(np.abs(a - b))


results = [
    ("低通结果", low_result),
    ("高通结果", high_result),
    ("带通结果", band_result),
]

plt.figure(figsize=(14, 7))

plt.subplot(2, 2, 1)
plt.imshow(noisy_image, cmap="gray", vmin=0, vmax=1)
plt.title("网络灰度图像")
plt.axis("off")

for i, (name, image) in enumerate(results):
    plt.subplot(2, 2, i + 2)
    plt.imshow(image, cmap="gray", vmin=0, vmax=1)
    change = mean_absolute_change(noisy_image, image)
    plt.title(f"{name}\n平均变化量={change:.4f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

print("低通结果平均变化量：", mean_absolute_change(noisy_image, low_result))
print("高通结果平均变化量：", mean_absolute_change(noisy_image, high_result))
print("带通结果平均变化量：", mean_absolute_change(noisy_image, band_result))

## 10. 理想滤波器与巴特沃斯滤波器对比

下面比较两类滤波器的区别：

1. **理想滤波器**：掩膜只有 0 和 1，截止边界非常清晰，频率被突然截断，容易带来振铃现象。
2. **巴特沃斯滤波器**：掩膜在 0 到 1 之间平滑变化，截止频率附近逐渐过渡，结果通常更柔和。

对比时重点观察：

1. 两类滤波器掩膜边界是否平滑。
2. 滤波结果是否出现明显边缘振荡或过渡突兀。
3. 平均变化量是否不同。

In [ ]:
comparison_items = [
    (
        "低通",
        ideal_low_mask,
        low_mask,
        ideal_low_result,
        low_result,
    ),
    (
        "高通",
        ideal_high_mask,
        high_mask,
        ideal_high_result,
        high_result,
    ),
    (
        "带通",
        ideal_band_mask,
        band_mask,
        ideal_band_result,
        band_result,
    ),
]

plt.figure(figsize=(15, 10))

for i, (name, ideal_mask, butter_mask, ideal_result, butter_result) in enumerate(comparison_items):
    plt.subplot(3, 5, i * 5 + 1)
    plt.imshow(ideal_mask, cmap="gray", vmin=0, vmax=1)
    plt.title(f"理想{name}掩膜")
    plt.axis("off")

    plt.subplot(3, 5, i * 5 + 2)
    plt.imshow(butter_mask, cmap="gray", vmin=0, vmax=1)
    plt.title(f"巴特沃斯{name}掩膜")
    plt.axis("off")

    plt.subplot(3, 5, i * 5 + 3)
    plt.imshow(ideal_result, cmap="gray", vmin=0, vmax=1)
    ideal_change = mean_absolute_change(noisy_image, ideal_result)
    plt.title(f"理想{name}结果\n变化量={ideal_change:.4f}")
    plt.axis("off")

    plt.subplot(3, 5, i * 5 + 4)
    plt.imshow(butter_result, cmap="gray", vmin=0, vmax=1)
    butter_change = mean_absolute_change(noisy_image, butter_result)
    plt.title(f"巴特沃斯{name}结果\n变化量={butter_change:.4f}")
    plt.axis("off")

    plt.subplot(3, 5, i * 5 + 5)
    plt.imshow(np.abs(ideal_result - butter_result), cmap="hot")
    plt.title("两者结果差异")
    plt.axis("off")

plt.tight_layout()
plt.show()

for name, _, _, ideal_result, butter_result in comparison_items:
    ideal_change = mean_absolute_change(noisy_image, ideal_result)
    butter_change = mean_absolute_change(noisy_image, butter_result)
    result_diff = mean_absolute_change(ideal_result, butter_result)
    print(f"{name}：理想变化量={ideal_change:.4f}，巴特沃斯变化量={butter_change:.4f}，两者结果差异={result_diff:.4f}")

## 11. 调整巴特沃斯低通截止半径观察滤波效果

In [ ]:
cutoff_values = [12, 20, 32, 48]

plt.figure(figsize=(12, 6))

for i, cutoff in enumerate(cutoff_values):
    mask = butterworth_low_pass_filter(shape, cutoff=cutoff, order=2)
    result, _ = apply_frequency_filter(noisy_image, mask)
    change = mean_absolute_change(noisy_image, result)

    plt.subplot(2, len(cutoff_values), i + 1)
    plt.imshow(mask, cmap="gray", vmin=0, vmax=1)
    plt.title(f"巴特沃斯低通\n半径={cutoff}")
    plt.axis("off")

    plt.subplot(2, len(cutoff_values), i + 1 + len(cutoff_values))
    plt.imshow(result, cmap="gray", vmin=0, vmax=1)
    plt.title(f"变化量={change:.4f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 12. 实验小结

本实验使用 `numpy.fft` 完成图像傅里叶变换和反变换，并自己实现了理想低通、高通、带通滤波器以及巴特沃斯低通、高通、带通滤波器的频域掩膜。

需要掌握的重点：

1. `np.fft.fft2` 可以将图像从空间域转换到频率域。
2. `np.fft.fftshift` 可以将低频移动到频谱中心。
3. 低通滤波器保留低频，通常能平滑图像并抑制高频细节，但会导致图像模糊。
4. 高通滤波器保留高频，能够突出边缘、纹理和细节。
5. 带通滤波器只保留指定频率范围，可以观察或提取特定频率结构。
6. 理想滤波器在截止频率处突然截断，掩膜边界硬，结果可能更容易出现振铃。
7. 巴特沃斯滤波器在截止频率附近平滑过渡，结果通常更柔和，但频率选择不如理想滤波器那么绝对。
8. 由于本实验使用的是网络图像，没有对应的干净真值图像，因此重点观察滤波前后视觉效果和频谱变化，而不是用误差指标判断去噪质量。

思考题：

1. 为什么巴特沃斯滤波器的结果通常比理想滤波器更柔和？
2. 巴特沃斯滤波器阶数增大时，结果会更接近哪一种滤波器？
3. 为什么高通滤波后图像边缘更明显，但整体结构变暗？
4. 如果要定量评价真实网络图像的去噪效果，需要额外准备什么数据？